# 01 - Data Understanding

**Goal:** understand the four competition files before any transformation: schema, coverage, missing values, duplicates, the train/test boundary and the main methodological risks.

**Source:** [Kaggle - Rossmann Store Sales](https://www.kaggle.com/competitions/rossmann-store-sales). Daily sales for 1,115 German drugstores, 2013-01-01 to 2015-07-31 (train) and a 48-day test horizon 2015-08-01 to 2015-09-17.

**Why this step matters:** every later modelling decision (which rows are predictable, which features are available at forecast time, how the metric treats closed days) follows from the facts established here.

In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 110
print("Project root:", PROJECT_ROOT)

Project root: rossmann-sales-forecasting


## 1. Load the raw files and inspect their shape

`StateHoliday` is read as text because the raw file mixes `0`, `a`, `b`, `c`. `Open` in `test.csv` is read as float because 11 rows are blank.

In [2]:
from src.data_preparation import load_raw_data
from src.config import load_config

cfg = load_config()
raw = load_raw_data(cfg)

for name, frame in raw.items():
    print(f"{name:11s} rows={frame.shape[0]:>8,}  columns={frame.shape[1]:>2}")
    display(frame.head(3))

train       rows=1,017,209  columns= 9


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1


test        rows=  41,088  columns= 8


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0


store       rows=   1,115  columns=10


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,<NA>
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"


submission  rows=  41,088  columns= 2


,Id,Sales
0,1,0
1,2,0
2,3,0


## 2. Schema, missing values and duplicates

In [3]:
train, test, store, submission = raw["train"], raw["test"], raw["store"], raw["submission"]

schema = pd.DataFrame({
    "train_dtype": train.dtypes.astype(str),
    "train_missing": train.isna().sum(),
    "test_dtype": test.dtypes.astype(str),
    "test_missing": test.isna().sum(),
})
display(schema)

summary = pd.DataFrame({
    "table": ["train", "test", "store", "sample_submission"],
    "rows": [len(train), len(test), len(store), len(submission)],
    "duplicate_store_date": [
        train.duplicated(["Store", "Date"]).sum(),
        test.duplicated(["Store", "Date"]).sum(),
        0, 0,
    ],
    "unique_stores": [train.Store.nunique(), test.Store.nunique(), store.Store.nunique(), 0],
})
display(summary)

,train_dtype,train_missing,test_dtype,test_missing
Customers,int64,0.0,NaN,NaN
Date,datetime64[us],0.0,datetime64[us],0.0
DayOfWeek,int8,0.0,int8,0.0
Id,NaN,NaN,int64,0.0
Open,int8,0.0,float64,11.0
Promo,int8,0.0,int8,0.0
Sales,int64,0.0,NaN,NaN
SchoolHoliday,int8,0.0,int8,0.0
StateHoliday,string,0.0,string,0.0
Store,int64,0.0,int64,0.0


,table,rows,duplicate_store_date,unique_stores
0,train,1017209,0,1115
1,test,41088,0,856
2,store,1115,0,1115
3,sample_submission,41088,0,0


## 3. Date coverage and the train/test boundary

The test horizon starts the day after the training data ends. This is a genuine **out-of-time** problem: all validation in this project uses chronological splits, never a random `train_test_split`.

In [4]:
coverage = pd.DataFrame({
    "split": ["train", "test"],
    "start": [train.Date.min(), test.Date.min()],
    "end": [train.Date.max(), test.Date.max()],
    "days": [(train.Date.max() - train.Date.min()).days + 1, (test.Date.max() - test.Date.min()).days + 1],
    "stores": [train.Store.nunique(), test.Store.nunique()],
    "rows": [len(train), len(test)],
})
display(coverage)

print("Test stores missing from train:", sorted(set(test.Store) - set(train.Store)))
print("Train stores absent from test :", len(set(train.Store) - set(test.Store)))
print("Test rows per store           :", test.Store.value_counts().unique())
print("Missing Open values in test   :", int(test.Open.isna().sum()))
print("Submission Ids match test Ids :", bool((submission.Id.values == test.Id.values).all()))

,split,start,end,days,stores,rows
0,train,2013-01-01,2015-07-31,942,1115,1017209
1,test,2015-08-01,2015-09-17,48,856,41088


Test stores missing from train: []
Train stores absent from test : 259
Test rows per store           : [48]
Missing Open values in test   : 11
Submission Ids match test Ids : True


**Interpretation.** The competition test set covers only 856 of the 1,115 stores. The remaining 259 stores were closed for renovation or permanently during the test period, so the model must not expect them. All 856 test stores appear in the training data, which means no cold-start problem. The submission file preserves the test Id order.

## 4. Target behaviour: closed days, zero sales and the metric

RMSPE divides by the actual value, so it is undefined for `Sales = 0`. Closed days therefore have to be predicted as zero by a business rule, and the metric is computed on open days with positive sales.

In [5]:
open_rows = train[train.Open == 1]
zero_open = train[(train.Open == 1) & (train.Sales == 0)]

pd.set_option("display.float_format", "{:,.2f}".format)
display(pd.DataFrame({
    "metric": ["rows", "closed store-days", "open store-days", "zero sales (all)", "zero sales while open",
               "mean sales (open)", "median sales (open)", "mean customers (open)", "sales per customer"],
    "value": [len(train), (train.Open == 0).sum(), len(open_rows), (train.Sales == 0).sum(), len(zero_open),
              open_rows.Sales.mean(), open_rows.Sales.median(), open_rows.Customers.mean(),
              open_rows.Sales.sum() / open_rows.Customers.sum()],
}))

print("Zero-sales-while-open rows are anomalies, not the norm:")
display(zero_open[["Store", "Date", "DayOfWeek", "Promo", "Customers", "Open"]].head(6))

,metric,value
0,rows,"1,017,209.00"
1,closed store-days,"172,817.00"
2,open store-days,"844,392.00"
3,zero sales (all),"172,871.00"
4,zero sales while open,54.00
5,mean sales (open),"6,955.51"
6,median sales (open),"6,369.00"
7,mean customers (open),762.73
8,sales per customer,9.12


Zero-sales-while-open rows are anomalies, not the norm:


,Store,Date,DayOfWeek,Promo,Customers,Open
86825,971,2015-05-15,5,0,0,1
142278,674,2015-03-26,4,0,0,1
196938,699,2015-02-05,4,1,0,1
322053,708,2014-10-01,3,1,0,1
330176,357,2014-09-22,1,0,0,1
340348,227,2014-09-11,4,0,0,1


**Business meaning.** About 17% of store-days are closed. Treating them as zero-sales demand would bias every average downward. The 54 open-but-zero rows are anomalies (likely data errors or temporary closures); they are kept in the data for transparency but excluded from model fitting and from the percentage metrics.

## 5. Store attributes and structural missingness

In [6]:
display(store.isna().sum().to_frame("missing").T)

display(store.StoreType.value_counts().rename("stores").to_frame())
display(store.Assortment.value_counts().rename("stores").to_frame())
display(pd.crosstab(store.StoreType, store.Assortment, margins=True))

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
missing,0,0,0,3,354,354,0,544,544,544


,stores
StoreType,
a,602
d,348
c,148
b,17


,stores
Assortment,
a,593
c,513
b,9


Assortment,a,b,c,All
StoreType,,,,
a,381,0,221,602
b,7,9,1,17
c,77,0,71,148
d,128,0,220,348
All,593,9,513,1115


**Interpretation of missing values**

| Variable | Missing | Nature | Decision |
|---|---:|---|---|
| `CompetitionDistance` | 3 | truly unknown | median imputation + `CompetitionDistanceMissing` flag |
| `CompetitionOpenSinceMonth/Year` | 354 | competition started before the store or unknown | keep unknown; `CompetitionOpenMonths = -1`, `HasCompetitionStart = 0` |
| `Promo2SinceWeek/Year`, `PromoInterval` | 544 | structural: `Promo2 = 0` | not imputed; `Promo2Active = 0` |
| `Open` in test | 11 | data gap | store's modal opening state + `OpenImputed` flag |

No arbitrary dates are invented for competitions or promotions that may never have existed.

## 6. Data-quality checks recorded by the pipeline

In [7]:
quality_path = PROJECT_ROOT / "outputs" / "metrics" / "data_quality.json"
if quality_path.exists():
    quality = json.loads(quality_path.read_text())
    display(pd.DataFrame(quality["checks"]))
    print("All checks passed:", quality["tests_passed"])
else:
    print("Run `python -m src.data_preparation` to generate the quality report.")

,check,passed,detail
0,row_count_preserved_by_join,True,LEFT JOIN must not duplicate rows
1,unique_store_date_train,True,one row per store-day
2,unique_store_date_test,True,one row per store-day
3,no_missing_store_type,True,all stores matched store.csv
4,non_negative_sales,True,Sales >= 0
5,date_ranges_contiguous,True,test starts the day after train
6,all_test_stores_in_train,True,cold-start stores require separate handling
7,submission_rows_match_test,True,"official test split contains 41,088 store-day ..."


All checks passed: True


## 7. Data dictionary

| Variable | Type | Meaning | Available at forecast time? |
|---|---|---|---|
| `Store` | int | store identifier | yes |
| `DayOfWeek` | int | 1 = Monday ... 7 = Sunday | yes |
| `Date` | date | trading day | yes |
| `Sales` | int | turnover in PLN (target) | **no** (only in training) |
| `Customers` | int | number of customers | **no** for future days |
| `Open` | 0/1 | store open | yes (11 test values imputed) |
| `Promo` | 0/1 | daily promotion | yes (scheduled in advance) |
| `StateHoliday` | 0/a/b/c | public holiday type | yes |
| `SchoolHoliday` | 0/1 | school holiday | yes |
| `StoreType` | a-d | store format | yes (static) |
| `Assortment` | a-c | assortment level | yes (static) |
| `CompetitionDistance` | m | distance to nearest competitor | yes (static) |
| `CompetitionOpenSince*` | month/year | when the competitor opened | yes (static) |
| `Promo2` | 0/1 | continuous promotion programme | yes (static) |
| `Promo2Since*` | week/year | when Promo2 started | yes (static) |
| `PromoInterval` | text | months with Promo2 promotions | yes (static) |

## 8. Methodological risks identified before modelling

1. **Data leakage through future sales or customers.** All historical features must be computed from information strictly before the forecast origin. This is enforced in `src/features.py` and covered by `tests/test_features.py`.
2. **Recursive error accumulation.** A 48-day recursive forecast feeds predictions back into lag features. A one-step linear model scored RMSPE 0.195 on the holdout but 0.38 when recursed; the project therefore uses direct multi-horizon features anchored at the forecast origin.
3. **Closed stores.** They are not demand, and they dominate Sundays. Closed days are predicted as zero and excluded from RMSPE.
4. **Zero-sales anomalies.** 54 open rows with zero sales would make RMSPE explode; they are excluded from fitting and scoring.
5. **Seasonal coverage.** Only 2.5 years of history: one full annual cycle plus a partial second. Annual seasonality can be described but not estimated with high confidence.
6. **2015 level shift.** The holdout summer is stronger than the same period in 2014; any model anchored on older seasonal averages will underpredict. This is quantified in notebook 08.
7. **Metric asymmetry.** RMSPE penalises over- and under-prediction differently in relative terms, and it is dominated by low-volume days. MAE, RMSE and WAPE are reported alongside it.
8. **Static attributes vs dynamic environment.** Competition distance is a snapshot, not a time series; no competitor openings/closures are observed during the period.